In [ ]:
!pip install torch transformers datasets peft trl accelerate bitsandbytes swanlab

In [ ]:
# To run this faster, one can implement flash attention 2 (optional)
# !pip install -U flash-attn --no-build-isolation

In [ ]:
!swanlab login

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

import swanlab
import torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
import gc
import os
import json
import time
import math
from torch.utils.data import DataLoader, Subset
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

model_id = "Qwen/Qwen2.5-3B"
dataset_name = "openai/gsm8k"
output_dir = "./qwen2.5-3b-distill-student"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/683 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

# Data Loading

### Original

In [ ]:
dataset_name = "openai/gsm8k"
dataset = load_dataset(dataset_name, 'main')
print(len(dataset.data['train']))
train_3000 = dataset['train'].select(range(3000))
print(len(train_3000))

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

7473
3000


In [ ]:
def process_data(examples):
    sources = []
    for q, a in zip(examples["question"], examples["answer"]):
        full_text = f"Question: {q}\nAnswer: {a}{tokenizer.eos_token}"
        sources.append(full_text)

    model_inputs = tokenizer(
        sources,
        max_length=512,
        truncation=True,
        padding="max_length"
    )

    labels = [ids.copy() for ids in model_inputs["input_ids"]]

    answer_marker = tokenizer.encode("\nAnswer:", add_special_tokens=False)

    for i in range(len(labels)):
        input_id_list = model_inputs["input_ids"][i]

        idx = -1
        for j in range(len(input_id_list) - len(answer_marker)):
            if input_id_list[j : j + len(answer_marker)] == answer_marker:
                idx = j + len(answer_marker)
                break

        if idx != -1:
            for k in range(idx):
                labels[i][k] = -100

        # [PAD] no loss
        for k in range(len(input_id_list)):
            if input_id_list[k] == tokenizer.pad_token_id:
                labels[i][k] = -100

    model_inputs["labels"] = labels
    return model_inputs

In [ ]:
tokenized_train_3000 = train_3000.map(
    process_data,
    batched=True,
    remove_columns=train_3000.column_names
)

Parameter 'function'=<function process_data at 0x7e6560e61800> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only shown once. Subsequent hashing failures won't be shown.


Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

### Distilled

In [ ]:
def load_distillation_data(file_path, tokenizer):
    data_list = []
    max_observed_tokens = 0

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip(): continue
            item = json.loads(line)

            # assemble the new distilled data
            reasoning = item['response']['reasoning']
            final_ans = item['response']['answer']
            full_text = (
                f"Question: {item['question']}\n"
                f"Answer: {reasoning}\n"
                f"#### {final_ans}{tokenizer.eos_token}"
            )

            # Count tokens
            tokens = tokenizer.encode(full_text, add_special_tokens=False)
            max_observed_tokens = max(max_observed_tokens, len(tokens))
            data_list.append({"text": full_text})

    final_max_length = 2 ** math.ceil(math.log2(max_observed_tokens))

    print(f"Max #(token) from dataset: {max_observed_tokens}") # Max(token)
    print(f"We'll set max_length as: {final_max_length}") # max_length

    distill_ds = Dataset.from_list(data_list)

    def tokenize_fn(examples):
        res = tokenizer(
            examples["text"],
            max_length=final_max_length,
            truncation=True,
            padding="max_length"
        )

        labels = []
        for ids in res["input_ids"]:
            label = [(l if l != tokenizer.pad_token_id else -100) for l in ids]
            labels.append(label)

        res["labels"] = labels
        return res

    return distill_ds.map(
        tokenize_fn,
        batched=True,
        remove_columns=["text"]
    )

In [ ]:
distill_dataset = load_distillation_data("success.jsonl", tokenizer)

# Model Training

### Configs

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)

In [ ]:
# 4bit Quantized model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

### Train Calls

In [ ]:
def cleanup():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

In [ ]:
def train_and_profile(model, tokenized_train_3000, model_name, num_epochs=1, batch_size=4, num_workers=4, lr=3e-5):
    best_val_loss = float('inf')

    # train, val split
    tokenized_train_3000.set_format(type="torch", columns=["input_ids", "labels"])
    val_size = 300
    train_size = len(tokenized_train_3000) - val_size

    train_ds = Subset(tokenized_train_3000, range(train_size))
    val_ds = Subset(tokenized_train_3000, range(train_size, len(tokenized_train_3000)))

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

    # optimizer/scheduler
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(train_loader) * num_epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1*total_steps), num_training_steps=total_steps)

    print(f"Starting Training: {model_name} | Train: {train_size} | Val: {val_size}")

    for epoch in range(num_epochs):
        epoch_train_loss = 0.0
        epoch_data_time = 0.0
        epoch_gpu_time = 0.0
        epoch_throughput = 0.0
        last_lr = 0.0

        # train
        model.train()
        start_data_time = time.time()

        pbar = tqdm(train_loader, desc=f"Epoch {epoch} [Train]")
        for batch in pbar:

            data_loading_duration = time.time() - start_data_time

            batch = {k: v.to(model.device) for k, v in batch.items()}

            torch.cuda.synchronize()
            start_gpu_time = time.time()

            optimizer.zero_grad()
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            optimizer.step()

            last_lr = scheduler.get_last_lr()[0]
            scheduler.step()

            torch.cuda.synchronize()
            gpu_duration = time.time() - start_gpu_time

            epoch_train_loss += loss.item()
            epoch_data_time += data_loading_duration
            epoch_gpu_time += gpu_duration
            epoch_throughput += (batch_size / (data_loading_duration + gpu_duration))

            pbar.set_postfix({"loss": f"{loss.item():.4f}"})
            start_data_time = time.time()

        cleanup()

        # eval
        model.eval()
        epoch_val_loss = 0.0
        with torch.no_grad():
            for val_batch in tqdm(val_loader, desc=f"Epoch {epoch} [Val]"):
                val_batch = {k: v.to(model.device) for k, v in val_batch.items()}
                val_outputs = model(**val_batch)
                epoch_val_loss += val_outputs.loss.item()

        cleanup()

        num_train_batches = len(train_loader)
        num_val_batches = len(val_loader)

        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss

            model.save_pretrained(last_model_path)
            tokenizer.save_pretrained(last_model_path)

            print(f"Saved best model so far.")

        swanlab.log({
            "epoch": epoch,
            "train/avg_loss": epoch_train_loss / num_train_batches,
            "train/avg_lr": last_lr,
            "val/avg_loss": epoch_val_loss / num_val_batches,
            "profile/avg_data_loading_time": epoch_data_time / num_train_batches,
            "profile/avg_gpu_compute_time": epoch_gpu_time / num_train_batches,
            "profile/avg_throughput": num_train_batches / epoch_gpu_time,
        }, step=epoch)

        print(f"Epoch {epoch} Summary: Train Loss: {epoch_train_loss/num_train_batches:.4f} | Val Loss: {epoch_val_loss/num_val_batches:.4f}")

    print(f"Training for {model_name} completed.")

### Original Data + No quantization

In [ ]:
model_standard = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    # attn_implementation="flash_attention_2",
)

In [ ]:
model_standard = get_peft_model(model_standard, lora_config)
model_standard.print_trainable_parameters()

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


In [ ]:
save_path = "./qwen2.5-3B-Orig-Orig"
last_model_path = save_path
swanlab.init(
    project="F-Qwen2.5-3B-Orig-Orig",
    experiment_name="F-Qwen2.5-3B-Orig-Orig")

swanlab: You have already initialized a run, the init function will be ignored

In [ ]:
train_and_profile(
    model_standard,
    tokenized_train_3000,
    num_epochs=7,
    model_name="Qwen2.5_3B_Orig_Orig",
    lr=2e-5,
    num_workers=16
)

Starting Training: Qwen2.5_3B_Orig_Orig | Train: 2700 | Val: 300


Epoch 0 [Val]: 100%|██████████| 75/75 [00:11<00:00,  6.36it/s]


Saved best model so far.
Epoch 0 Summary: Train Loss: 0.2980 | Val Loss: 0.2432


Epoch 1 [Val]: 100%|██████████| 75/75 [00:11<00:00,  6.37it/s]


Saved best model so far.
Epoch 1 Summary: Train Loss: 0.2313 | Val Loss: 0.2405


...

Epoch 4 Summary: Train Loss: 0.1262 | Val Loss: 0.3285


Epoch 5 [Val]: 100%|██████████| 75/75 [00:11<00:00,  6.35it/s]


Epoch 5 Summary: Train Loss: 0.1044 | Val Loss: 0.3713


Epoch 6 [Val]: 100%|██████████| 75/75 [00:11<00:00,  6.34it/s]


Epoch 6 Summary: Train Loss: 0.0911 | Val Loss: 0.4097
Training for Qwen2.5_3B_Orig_Orig completed.


### Distilled Data + No quantization

In [ ]:
model_distill = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    # attn_implementation="flash_attention_2",
)
model_distill = get_peft_model(model_distill, lora_config)
model_distill.print_trainable_parameters()
save_path = "./qwen2.5-3B-Distill-Orig"
last_model_path = save_path
swanlab.Run.finish()
swanlab.init(
    project="F-Qwen2.5-3B-Distill-Orig",
    experiment_name="Qwen2.5-3B-Distill-Orig"
)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


swanlab: 🏠 View project at https://swanlab.cn/@leow1728/F-Qwen2.5-3B-Orig-Orig

swanlab: 🚀 View run at https://swanlab.cn/@leow1728/F-Qwen2.5-3B-Orig-Orig/runs/pwdj07oszzum8233oe31d

Output()

...

swanlab: Run data will be saved locally in /content/swanlog/run-20260404_095354-l0fu2ba3utz656faik9i4

swanlab: 👋 Hi leow1728,welcome to swanlab!

swanlab: Syncing run Qwen2.5-3B-Distill-Orig to the cloud

swanlab: 🏠 View project at https://swanlab.cn/@leow1728/F-Qwen2.5-3B-Distill-Orig

swanlab: 🚀 View run at https://swanlab.cn/@leow1728/F-Qwen2.5-3B-Distill-Orig/runs/l0fu2ba3utz656faik9i4

In [ ]:
train_and_profile(
    model_distill,
    distill_dataset,
    num_epochs=7,
    model_name="Qwen2.5_3B_Distill_Orig",
    lr=2e-5,
    num_workers=16
)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 12, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Starting Training: Qwen2.5_3B_Distill_Orig | Train: 2700 | Val: 300


Epoch 0 [Train]:   0%|          | 0/675 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 12, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Epoch 0 [Val]: 100%|██████████| 75/75 [00:11<00:00,  6.32it/s]


Saved best model so far.
Epoch 0 Summary: Train Loss: 0.3412 | Val Loss: 0.2593


Epoch 1 [Val]: 100%|██████████| 75/75 [00:11<00:00,  6.33it/s]


...

Epoch 4 Summary: Train Loss: 0.1559 | Val Loss: 0.2950


Epoch 5 [Val]: 100%|██████████| 75/75 [00:11<00:00,  6.31it/s]


Epoch 5 Summary: Train Loss: 0.1332 | Val Loss: 0.3264


Epoch 6 [Val]: 100%|██████████| 75/75 [00:11<00:00,  6.33it/s]


Epoch 6 Summary: Train Loss: 0.1178 | Val Loss: 0.3604
Training for Qwen2.5_3B_Distill_Orig completed.


### Original Data + Quantization

In [ ]:
model_quantized = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={"": 0},
    trust_remote_code=True
)

model_quantized = prepare_model_for_kbit_training(model_quantized)
model_quantized = get_peft_model(model_quantized, lora_config)

model_quantized.print_trainable_parameters()

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


In [ ]:
save_path = "./qwen2.5-3B-Orig-Quant"
last_model_path = save_path
swanlab.Run.finish()
swanlab.init(
    project="F-Qwen2.5-3B-Orig-Quant",
    experiment_name="Qwen2.5-3B-Orig-Quant-Final"
)

swanlab: Tracking run with swanlab version 0.7.14

swanlab: Run data will be saved locally in /content/swanlog/run-20260404_102330-q1x7qd7rkh92t7tkqfxsh

swanlab: 👋 Hi leow1728,welcome to swanlab!

swanlab: Syncing run Qwen2.5-3B-Orig-Quant-Final to the cloud

swanlab: 🏠 View project at https://swanlab.cn/@leow1728/F-Qwen2.5-3B-Orig-Quant

swanlab: 🚀 View run at https://swanlab.cn/@leow1728/F-Qwen2.5-3B-Orig-Quant/runs/q1x7qd7rkh92t7tkqfxsh

In [ ]:
train_and_profile(
    model_quantized,
    tokenized_train_3000,
    num_epochs=7,
    model_name="Qwen2.5_3B_Distill_Orig",
    lr=2e-5,
    num_workers=16
)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 12, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Starting Training: Qwen2.5_3B_Distill_Orig | Train: 2700 | Val: 300


Epoch 0 [Train]:   0%|          | 0/675 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 12, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reen

swanlab: swanlab api error ('Trace id: None', 'POST https://api.swanlab.cn/api/house/metrics', '522 Connect origin 
timed out')

Epoch 0 [Train]:  99%|█████████▉| 670/675 [09:06<00:04,  1.23it/s, loss=0.3612]

...

Epoch 4 Summary: Train Loss: 0.1984 | Val Loss: 0.4034


Epoch 5 [Val]: 100%|██████████| 75/75 [00:21<00:00,  3.52it/s]


Epoch 5 Summary: Train Loss: 0.1720 | Val Loss: 0.4463


Epoch 6 [Val]: 100%|██████████| 75/75 [00:21<00:00,  3.51it/s]


Epoch 6 Summary: Train Loss: 0.1521 | Val Loss: 0.4864
Training for Qwen2.5_3B_Distill_Orig completed.


### Distilled Data + Quantization

In [ ]:
model_quantized_distill = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={"": 0},
    trust_remote_code=True
)

model_quantized_distill = prepare_model_for_kbit_training(model_quantized_distill)
model_quantized_distill = get_peft_model(model_quantized_distill, lora_config)

model_quantized_distill.print_trainable_parameters()

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


In [ ]:
save_path = "./qwen2.5-3B-Distill-Quant"
last_model_path = save_path
swanlab.Run.finish()
swanlab.init(
    project="F-Qwen2.5-3B-Distill-Quant",
    experiment_name="Qwen2.5-3B-Distill-Quant"
)

swanlab: Tracking run with swanlab version 0.7.14

swanlab: Run data will be saved locally in /content/swanlog/run-20260404_113032-4m7klczolzd6q77k016ms

swanlab: 👋 Hi leow1728,welcome to swanlab!

swanlab: Syncing run Qwen2.5-3B-Distill-Quant to the cloud

swanlab: 🏠 View project at https://swanlab.cn/@leow1728/F-Qwen2.5-3B-Distill-Quant

swanlab: 🚀 View run at https://swanlab.cn/@leow1728/F-Qwen2.5-3B-Distill-Quant/runs/4m7klczolzd6q77k016ms

In [ ]:
train_and_profile(
    model_quantized_distill,
    distill_dataset,
    num_epochs=7,
    model_name="Qwen2.5_3B_Distill_Quant",
    lr=2e-5,
    num_workers=16
)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 12, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Starting Training: Qwen2.5_3B_Distill_Quant | Train: 2700 | Val: 300


Epoch 0 [Train]:   0%|          | 0/675 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 12, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
 

Saved best model so far.
Epoch 0 Summary: Train Loss: 0.4667 | Val Loss: 0.3521


Epoch 1 [Val]: 100%|██████████| 75/75 [00:21<00:00,  3.50it/s]


...

Epoch 5 Summary: Train Loss: 0.1852 | Val Loss: 0.4029


Epoch 6 [Train]:  39%|███▉      | 263/675 [03:35<05:36,  1.23it/s, loss=0.1777]

swanlab: swanlab api error ('Trace id: None', 'POST https://api.swanlab.cn/api/house/metrics', '522 Connect origin 
timed out')

Epoch 6 [Val]: 100%|██████████| 75/75 [00:21<00:00,  3.47it/s]


Epoch 6 Summary: Train Loss: 0.1709 | Val Loss: 0.4263
Training for Qwen2.5_3B_Distill_Quant completed.


# Model Reloading

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

model_id = "Qwen/Qwen2.5-3B"
distill_lora_path = "./path-to-weights-with-distill" # path to the folder that contains weights of "XXX-Distill-XXX" models
orig_lora_path = "./path-to-weights" # path to the folder that contains weights of "XXX-Orig-XXX" models

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

# example: model_orig_orig
model_1 = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    # attn_implementation="flash_attention_2"
)

model_1 = PeftModel.from_pretrained(model_1, distill_lora_path)
model_1.eval()

# example: model_distill_quant
model_2 = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config, # use this line in AutoModelForCausalLM for quantized models
    device_map={"": 0},
    trust_remote_code=True
)

model_2 = PeftModel.from_pretrained(model_2, orig_lora_path) # orig_lora_path for model_distill_quant may be 'Qwen2.5-3B-Distill-Quant'
model_2.eval()


# Evaluate on dataset

In [ ]:
model_standard = torch.compile(model_standard)

In [ ]:
import re

def extract_answer(text):
    gsm_match = re.findall(r"####\s*(-?\d+(?:\.\d+)?)", text)
    if gsm_match:
        return gsm_match[-1].strip()

    numbers = re.findall(r"-?\d+(?:\.\d+)?", text)
    return numbers[-1] if numbers else ""

def is_correct(pred_str, ground_truth_str):
    pred = extract_answer(pred_str)
    gt = extract_answer(ground_truth_str)
    return pred == gt

In [ ]:
@torch.no_grad()
def evaluate_model(model, tokenizer, test_dataset, model_name, batch_size=8):
    model.eval()
    correct_count = 0
    total_count = len(test_dataset)

    results = []

    torch.cuda.synchronize()
    start_time = time.time()

    print(f"Evaluating {model_name}...")

    for i in tqdm(range(0, total_count, batch_size)):
        batch_data = test_dataset[i : i + batch_size]

        prompts = [f"Question: {q}\nAnswer: " for q in batch_data["question"]]

        inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

        for j, output in enumerate(outputs):
            full_decode = tokenizer.decode(output, skip_special_tokens=True)
            generated_text = full_decode[len(prompts[j]):]

            gt_answer = batch_data["answer"][j]

            if i % 10 == 0 and j == 0:
                print(f"Question: {batch_data['question'][j]}")
                print(f"Generated Answer: {generated_text}")
                print(f"Ground Truth Answer: {gt_answer}")

            if is_correct(generated_text, gt_answer):
                correct_count += 1

            results.append({
                "question": batch_data["question"][j],
                "pred": generated_text,
                "gt": gt_answer
            })

    torch.cuda.synchronize()
    total_duration = time.time() - start_time

    accuracy = correct_count / total_count
    samples_per_sec = total_count / total_duration

    print(f"\n--- Result: {model_name} ---")
    print(f"Accuracy: {accuracy:.2%}")
    print(f"Inference Speed: {samples_per_sec:.2f} samples/s")
    print(f"Total Time: {total_duration:.2f}s")

    return accuracy, results

In [ ]:
import json

In [ ]:
test_dataset = dataset["test"]
test_tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if test_tokenizer.pad_token is None:
    test_tokenizer.pad_token = test_tokenizer.eos_token
test_tokenizer.padding_side = "left"

acc_std, res_std = evaluate_model(
    model_standard,
    test_tokenizer,
    test_dataset,
    "Standard_BF16_LoRA"
)

with open("eval_results.json", "w") as f:
    json.dump({"standard": res_std}, f, indent=4)

Evaluating Standard_BF16_LoRA...


  1%|          | 1/165 [00:22<1:01:15, 22.41s/it]

Question: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
Generated Answer: 16 - 3 - 4 = <<16-3-4=9>>9 eggs
9 * 2 = $<<9*2=18>>18 per day
#### 18
Ground Truth Answer: Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18


  4%|▎         | 6/165 [02:12<56:47, 21.43s/it]  

Question: Brandon's iPhone is four times as old as Ben's iPhone. Ben's iPhone is two times older than Suzy's iPhone. If Suzy’s iPhone is 1 year old, how old is Brandon’s iPhone?
Generated Answer: 1 year x 2 = <<1*2=2>>2 years old is Ben's iPhone.
2 years old x 4 = <<2*4=8>>8 years old is Brandon's iPhone.
#### 8
Ground Truth Answer: Ben’s iPhone is 1*2 = <<1*2=2>>2 years old.
Brandon’s iPhone is 4*2 = <<4*2=8>>8 years old.
#### 8


...

Question: Elly is organizing her books on the new bookcases her parents bought her. Each of the middle 2 shelves can hold 10 books. The bottom shelf can hold twice as many books as a middle shelf. The top shelf can hold 5 fewer books than the bottom shelf. If she has 110 books, how many bookcases does she need to hold all of them?
Generated Answer: 110 / (10 + 10 + 2 * 10 + 10 + 5) = 110 / 55 books per shelf = <<110/(10+10+2*10+10+5)=2>>2 bookcases
#### 2
Ground Truth Answer: The bottom shelf can hold 2 * 10 = <<2*10=20>>20 books.
The 2 middle shelves can hold a total of 2 * 10 = <<2*10=20>>20 books.
The top shelf can hold 20 – 5 = <<20-5=15>>15 books.
Each bookcase can hold a total of 20 books + 20 books + 15 books = <<20+20+15=55>>55 books.
To hold all of her books, she needs 110 / 55 = <<110/55=2>>2 bookcases.
#### 2


 98%|█████████▊| 161/165 [1:01:08<01:41, 25.32s/it]

Question: Steve put together a puzzle that took 10 hours of hard work to complete. Anna put together the same puzzle in 2 hours more than half Steve's time. How long did it take Anna to finish the difficult puzzle?
Generated Answer: 10 / 2 = <<10/2=5>>5 hours
5 + 2 = <<5+2=7>>7 hours
Anna spent 7 hours on the puzzle.
#### 7
Ground Truth Answer: Half the time it took Steve to put together the puzzle is 10/2=<<10/2=5>>5
If Anna put together the same puzzle in 2 hours more than half the time it took Steve, she took 5+2=7 hours to finish the puzzle.
#### 7


100%|██████████| 165/165 [1:02:32<00:00, 22.74s/it]


--- Result: Standard_BF16_LoRA ---
Accuracy: 58.45%
Inference Speed: 0.35 samples/s
Total Time: 3752.03s
